# RoadGuard AI — automatic Swin Transformer tuning
This notebook runs a small GPU-based search for Swin-T + FPN + Faster R-CNN on the four RDD2022 classes D00/D10/D20/D40. It selects a checkpoint using validation mAP, tunes confidence thresholds on validation only, and evaluates the held-out test split once. **It does not guarantee higher metrics.**

Before starting, choose **Runtime → Change runtime type → GPU**. Outputs and the final checkpoint will be copied to Google Drive.

In [ ]:
import torch, sys, subprocess, pathlib, os, json, shutil, zipfile, urllib.request, xml.etree.ElementTree as ET, hashlib
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Colab, then rerun this cell.')
print('Python:',sys.version,'PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))

!pip -q install -U openmim
!pip -q uninstall -y mmcv mmcv-lite
!mim install 'mmengine>=0.7.1,<1.0.0'
!mim install 'mmcv>=2.0.0,<2.2.0'
REPO=Path('/content/roadguard-ai')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','feat/swin-transformer-roadguard','https://github.com/vishal122006/roadguard-ai.git',str(REPO)],check=True)
MMDET=Path('/content/mmdetection')
if not (MMDET/'.git').exists():
    subprocess.run(['git','clone','--depth','1','--branch','v3.3.0','https://github.com/open-mmlab/mmdetection.git',str(MMDET)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(MMDET)],check=True)
!pip -q install pycocotools pillow
import mmcv,mmengine,mmdet
from mmcv.ops import nms
print('MMDetection:',mmdet.__version__,'MMCV:',mmcv.__version__,'MMEngine:',mmengine.__version__,'MMCV CUDA ops OK')
if not mmdet.__version__.startswith('3.3.'):
    raise RuntimeError('This notebook expects MMDetection 3.3.x.')
CONFIG_SRC=REPO/'configs/roadguard_swin_faster_rcnn.py'
CONFIG_DST=MMDET/'configs/roadguard_swin_faster_rcnn.py'
shutil.copy2(CONFIG_SRC,CONFIG_DST)
shutil.copy2(REPO/'configs/roadguard_swin_faster_rcnn_baseline.py',MMDET/'configs/roadguard_swin_faster_rcnn_baseline.py')
PROJECT=REPO
print('Repository:',REPO,'MMDetection:',MMDET)

## 2. Download RDD2022 India and pair real image/XML annotations
This cell skips ambiguous image/XML matches and reports pairing counts. Review the printed counts. It uses only real labelled images; it does not generate synthetic detections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_URL='https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_India.zip'
ZIP=Path('/content/RDD2022_India.zip'); RAW=Path('/content/rdd2022_india_raw')
if not ZIP.exists() or ZIP.stat().st_size < 10_000_000:
    print('Downloading official RDD2022 India archive…')
    urllib.request.urlretrieve(DATA_URL,ZIP)
if not RAW.exists() or not any(RAW.iterdir()):
    RAW.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(ZIP) as zf:
        bad=zf.testzip()
        if bad: raise RuntimeError(f'Corrupt ZIP member: {bad}')
        zf.extractall(RAW)
exts={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
all_images=[p for p in RAW.rglob('*') if p.is_file() and p.suffix.lower() in exts]
all_xml=[p for p in RAW.rglob('*.xml') if p.is_file()]
from collections import defaultdict
by_name,by_stem=defaultdict(list),defaultdict(list)
for img in all_images: by_name[img.name.casefold()].append(img); by_stem[img.stem.casefold()].append(img)
SRC=Path('/content/RDD2022_paired')
if SRC.exists(): shutil.rmtree(SRC)
(SRC/'images').mkdir(parents=True); (SRC/'annotations').mkdir(parents=True)
paired=ambiguous=unmatched=bad_images=0; seen=set()
from PIL import Image
for xp in all_xml:
    try: root=ET.parse(xp).getroot()
    except Exception: continue
    fname=(root.findtext('filename') or '').replace('\\','/').strip()
    candidates=by_name.get(Path(fname).name.casefold(),[]) if fname else []
    if not candidates: candidates=by_name.get((xp.stem+'.jpg').casefold(),[])+by_name.get((xp.stem+'.jpeg').casefold(),[])+by_name.get((xp.stem+'.png').casefold(),[])
    if not candidates: candidates=by_stem.get(Path(fname).stem.casefold(),[]) if fname else by_stem.get(xp.stem.casefold(),[])
    same=[p for p in candidates if p.parent==xp.parent]
    if len(same)==1: candidates=same
    if len(candidates)>1:
        xt=set(xp.relative_to(RAW).parts[:-1]); ranked=[(len(xt.intersection(set(p.relative_to(RAW).parts[:-1]))),p) for p in candidates]
        best=max(v for v,_ in ranked); candidates=[p for v,p in ranked if v==best] if best else candidates
    if len(candidates)!=1:
        if candidates: ambiguous+=1
        else: unmatched+=1
        continue
    img=candidates[0]
    if str(img.resolve()) in seen: continue
    try:
        with Image.open(img) as im: im.verify()
    except Exception: bad_images+=1; continue
    rel=img.relative_to(RAW); dst=SRC/'images'/rel; ann=SRC/'annotations'/rel.with_suffix('.xml')
    dst.parent.mkdir(parents=True,exist_ok=True); ann.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(img,dst); shutil.copy2(xp,ann); seen.add(str(img.resolve())); paired+=1
print({'all_images':len(all_images),'xml_files':len(all_xml),'paired':paired,'ambiguous_skipped':ambiguous,'unmatched':unmatched,'unreadable_images':bad_images})
if paired < 30: raise RuntimeError('Too few unambiguous pairs. Inspect the archive/XML paths; do not train on guessed matches.')

## 3. Convert to COCO and audit
The converter creates reproducible 70/15/15 train/validation/test splits. If the archive contains adjacent frames or route-related images, inspect source paths and update the converter's group key before relying on the metrics.

In [ ]:
DATA_ROOT=MMDET/'data/RDD2022_coco'
subprocess.run([sys.executable,str(REPO/'scripts/convert_voc_to_coco_swin.py'),'--images',str(SRC/'images'),'--annotations',str(SRC/'annotations'),'--out',str(DATA_ROOT),'--seed','42'],check=True)
subprocess.run([sys.executable,str(REPO/'scripts/check_dataset.py'),'--root',str(DATA_ROOT)],check=True)
print((DATA_ROOT/'dataset_summary.json').read_text()[:3000])
print((DATA_ROOT/'dataset_check_summary.json').read_text()[:3000])

## 4. Automatic hyperparameter search and fine-tuning
Default setup runs 3 candidates × 8 epochs, then fine-tunes the validation winner for up to 24 more epochs. This may take several hours or more depending on dataset size/GPU. Lower the batch size/scales if CUDA memory is exhausted.

In [ ]:
OUT=MMDET/'work_dirs/roadguard_autotune'
SEARCH_EPOCHS=8
FINAL_EPOCHS=24
cmd=[sys.executable,str(REPO/'scripts/auto_tune_swin.py'),'--config',str(CONFIG_DST),'--mmdet-root',str(MMDET),'--data-root',str(DATA_ROOT),'--out-dir',str(OUT),'--search-epochs',str(SEARCH_EPOCHS),'--final-epochs',str(FINAL_EPOCHS)]
subprocess.run(cmd,check=True,cwd=MMDET)
SUMMARY=json.loads((OUT/'autotune_summary.json').read_text())
CONFIG=Path(SUMMARY['final_config']); CHECKPOINT=Path(SUMMARY['final_checkpoint']); FINAL_WORK=Path(SUMMARY['final_work_dir'])
print('Selected trial:',SUMMARY['selected_search_trial']['name'])
print('Validation mAP:',SUMMARY['final_validation_coco_bbox_mAP'])
print('Checkpoint:',CHECKPOINT)

## 5. Tune class thresholds on validation only
The threshold sweep selects the highest-recall threshold meeting a target precision where possible. A class that cannot reach the target uses its validation F1 maximum. Threshold changes trade recall for precision and do not retrain model weights.

In [ ]:
THRESHOLD_DIR=FINAL_WORK/'threshold_analysis'; THRESHOLD_DIR.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,str(REPO/'scripts/tune_swin_thresholds.py'),'--config',str(CONFIG),'--checkpoint',str(CHECKPOINT),'--data-root',str(DATA_ROOT),'--split','val','--mode','tune','--step','0.025','--out',str(THRESHOLD_DIR)],check=True,cwd=MMDET)
import csv
TARGET_PRECISION=0.70
sweep=list(csv.DictReader((THRESHOLD_DIR/'validation_threshold_sweep.csv').open(newline='',encoding='utf-8')))
td=json.loads((THRESHOLD_DIR/'best_thresholds.json').read_text())
chosen={}; status={}; selected_rows={}
for cls in ['D00','D10','D20','D40']:
    rows=[r for r in sweep if r['class']==cls]
    for r in rows:
        for k in ['threshold','tp','fp','fn','precision','recall','f1']: r[k]=float(r[k])
    acceptable=[r for r in rows if r['tp']>0 and r['precision']>=TARGET_PRECISION]
    if acceptable: pick=max(acceptable,key=lambda r:(r['recall'],r['f1'],r['precision'])); status[cls]=True
    else: pick=max(rows,key=lambda r:(r['f1'],r['precision'],r['recall'])); status[cls]=False
    chosen[cls]=float(pick['threshold']); selected_rows[cls]=pick
td['per_class_thresholds']=chosen; td['requested_minimum_precision']=TARGET_PRECISION
td['precision_target_reached_by_class']=status; td['per_class_validation_metrics_at_selected_threshold']=selected_rows
td['selection_rule']='Validation only: highest recall among thresholds reaching target precision; otherwise best validation F1. Freeze before test.'
(THRESHOLD_DIR/'best_thresholds.json').write_text(json.dumps(td,indent=2))
print('Frozen thresholds:',chosen); print('Reached validation target:',status)
if not all(status.values()): print('At least one class did not reach target precision on validation; its threshold falls back to best F1.')

## 6. Final test evaluation — no further tuning
Do not change the checkpoint or thresholds after viewing test results. COCO mAP and threshold-specific precision/recall/F1 measure different things and should be reported separately.

In [ ]:
TEST_OUT=FINAL_WORK/'final_test'; TEST_OUT.mkdir(parents=True,exist_ok=True)
result=subprocess.run([sys.executable,'tools/test.py',str(CONFIG),str(CHECKPOINT)],cwd=MMDET,capture_output=True,text=True)
(TEST_OUT/'mmdet_test_log.txt').write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
print(result.stdout[-7000:]); print(result.stderr[-2000:] if result.stderr else '')
if result.returncode: raise RuntimeError('Held-out test evaluation failed; inspect final_test/mmdet_test_log.txt')
subprocess.run([sys.executable,str(REPO/'scripts/tune_swin_thresholds.py'),'--config',str(CONFIG),'--checkpoint',str(CHECKPOINT),'--data-root',str(DATA_ROOT),'--split','test','--mode','evaluate','--thresholds-json',str(THRESHOLD_DIR/'best_thresholds.json'),'--out',str(TEST_OUT)],check=True,cwd=MMDET)
print((TEST_OUT/'test_metrics_frozen_thresholds.csv').read_text())

## 7. Benchmark and save to Google Drive
The checkpoint is saved separately from the existing YOLO weights. Review the test metrics before switching the deployed app.

In [ ]:
BENCH=FINAL_WORK/'benchmark'
subprocess.run([sys.executable,str(REPO/'scripts/benchmark_swin.py'),'--config',str(CONFIG),'--checkpoint',str(CHECKPOINT),'--source',str(DATA_ROOT/'images'),'--max-images','50','--out',str(BENCH/'inference_benchmark.json')],check=True,cwd=MMDET)
DRIVE_OUT=Path('/content/drive/MyDrive/RoadGuardAI_SwinTransformer')
(DRIVE_OUT/'checkpoints').mkdir(parents=True,exist_ok=True); (DRIVE_OUT/'reports').mkdir(parents=True,exist_ok=True)
shutil.copy2(CHECKPOINT,DRIVE_OUT/'checkpoints/roadguard_swin.pth')
shutil.copy2(THRESHOLD_DIR/'best_thresholds.json',DRIVE_OUT/'checkpoints/best_thresholds.json')
for folder in [OUT,THRESHOLD_DIR,TEST_OUT,BENCH,DATA_ROOT]:
    if folder.exists():
        for item in folder.rglob('*'):
            if item.is_file() and item.suffix.lower() in {'.csv','.json','.txt'}:
                shutil.copy2(item,DRIVE_OUT/'reports'/item.name)
print('Saved model and reports to',DRIVE_OUT)
print('Compare the measured test metrics with the prior model under the same split/protocol before claiming improvement.')

## 8. Automatically update the Streamlit performance tracker
This publishes the measured validation mAP and final test precision/recall/F1 to `results/swin_tracking.json` on GitHub `main`, which triggers Streamlit Cloud to redeploy if connected to that branch. It is optional. Before training, create a Colab Secret named `ROADGUARD_GITHUB_TOKEN` with Contents read/write permission for the repository. Only compact metrics are committed—never your checkpoint or dataset. If the secret is absent, the results are saved to Drive and the publish step is skipped. The 70% target stays labelled as a goal, not an achieved score.

In [ ]:
# 8. Publish the measured Swin metrics to the live Streamlit dashboard (optional automatic step)
# One-time setup: in Colab, open the key icon (Secrets) and add ROADGUARD_GITHUB_TOKEN.
# The token needs Contents: Read and write access for vishal122006/roadguard-ai.
# This publishes only results/swin_tracking.json to main; it does not upload model weights or rewrite app code.
import base64, csv, json, urllib.error, urllib.request
from datetime import datetime, timezone
from google.colab import userdata
validation_map = float(SUMMARY['final_validation_coco_bbox_mAP'])
validation_map_percent = validation_map * 100.0 if abs(validation_map) <= 1.0 else validation_map
test_rows = list(csv.DictReader((TEST_OUT/'test_metrics_frozen_thresholds.csv').open(newline='', encoding='utf-8')))
test_by_name = {row.get('class'): row for row in test_rows}
def as_percent(row, key):
    try: return round(float(row[key]) * 100.0, 2)
    except (KeyError, TypeError, ValueError): return None
micro = test_by_name.get('MICRO_AGGREGATE', {})
macro = test_by_name.get('MACRO_AVERAGE', {})
published_metrics = {
    'model': 'Swin-Tiny + FPN + Faster R-CNN',
    'stage': 'fine-tuned candidate',
    'epoch': int(FINAL_EPOCHS),
    'split': 'validation',
    'metric_name': 'COCO mAP@0.50:0.95 (validation)',
    'metric_value_percent': round(validation_map_percent, 2),
    'target_percent': 70.0,
    'fine_tuned_evaluated': True,
    'last_updated_utc': datetime.now(timezone.utc).isoformat(),
    'source': 'RoadGuard AI Colab AutoTune',
    'test_metrics': {
        'precision_micro_percent': as_percent(micro, 'precision'),
        'recall_micro_percent': as_percent(micro, 'recall'),
        'f1_micro_percent': as_percent(micro, 'f1'),
        'precision_macro_percent': as_percent(macro, 'precision'),
        'recall_macro_percent': as_percent(macro, 'recall'),
        'f1_macro_percent': as_percent(macro, 'f1')
    },
    'note': 'Measured validation mAP from the fine-tuned candidate. Target 70% remains a goal, not a guaranteed or assumed result.'
}
LOCAL_TRACKING = DRIVE_OUT/'reports'/'swin_tracking.json'
LOCAL_TRACKING.write_text(json.dumps(published_metrics, indent=2), encoding='utf-8')
print('Saved measured tracking JSON locally:', LOCAL_TRACKING)
try:
    github_token = userdata.get('ROADGUARD_GITHUB_TOKEN')
except Exception:
    github_token = None
if not github_token:
    print('Automatic GitHub publishing skipped: add a Colab Secret named ROADGUARD_GITHUB_TOKEN with Contents read/write permission to vishal122006/roadguard-ai, then rerun this cell.')
else:
    api = 'https://api.github.com/repos/vishal122006/roadguard-ai/contents/results/swin_tracking.json'
    headers = {'Authorization': f'Bearer {github_token}', 'Accept': 'application/vnd.github+json', 'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'RoadGuard-AutoTune'}
    existing_sha = None
    get_req = urllib.request.Request(api + '?ref=main', headers=headers, method='GET')
    try:
        with urllib.request.urlopen(get_req, timeout=30) as response:
            existing = json.loads(response.read().decode('utf-8'))
            existing_sha = existing.get('sha')
    except urllib.error.HTTPError as exc:
        if exc.code != 404: raise RuntimeError(f'Could not read existing GitHub metric report (HTTP {exc.code}); check token permission.')
    body = {'message': 'Auto-update measured Swin validation metrics from Colab', 'content': base64.b64encode(json.dumps(published_metrics, indent=2).encode('utf-8')).decode('ascii'), 'branch': 'main'}
    if existing_sha: body['sha'] = existing_sha
    put_req = urllib.request.Request(api, data=json.dumps(body).encode('utf-8'), headers={**headers, 'Content-Type': 'application/json'}, method='PUT')
    try:
        with urllib.request.urlopen(put_req, timeout=30) as response:
            published = json.loads(response.read().decode('utf-8'))
        print('Published measured metrics to GitHub main. Commit:', published.get('commit', {}).get('sha'))
        print('If Streamlit Cloud is connected to vishal122006/roadguard-ai → main → app.py, it will redeploy and the Swin tracker will show the new result.')
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')[:1000]
        raise RuntimeError(f'GitHub metrics publish failed (HTTP {exc.code}). Verify ROADGUARD_GITHUB_TOKEN Contents read/write permission. Details: {detail}')
